In [1]:
from data.preprocessing.pipeline import Pipeline

In [2]:
pipeline = Pipeline(download=False)

Extracting raw features: 100%|███████████████████████████████████████████████████████| 888/888 [00:05<00:00, 154.16it/s]


In [6]:
import pandas as pd
import numpy as np
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report

from catboost import CatBoostClassifier

# -------------------------------------------------------
# 1. Загрузка данных
# -------------------------------------------------------
train_df, test_df = pipeline.get_processed_data()

y = train_df["taste_cluster"].values
feature_cols = [c for c in train_df.columns if c.startswith("feat_")]
X = train_df[feature_cols].values
ids = train_df["Pubchem_ID"].values

# -------------------------------------------------------
# 2. Split с сохранением Pubchem_ID
# -------------------------------------------------------
X_train, X_val, y_train, y_val, ids_train, ids_val = train_test_split(
    X, y, ids,
    test_size=0.2,
    stratify=y,
    random_state=42
)

# -------------------------------------------------------
# 3. Черновой CatBoost → отбор признаков
# -------------------------------------------------------
print("\n>>> Быстрая модель CatBoost для отбора признаков...")

tmp_model = CatBoostClassifier(
    loss_function="MultiClass",
    eval_metric="TotalF1",
    iterations=400,
    learning_rate=0.1,
    depth=6,
    verbose=False,
    random_seed=42
)

tmp_model.fit(X_train, y_train)

importances = np.array(tmp_model.get_feature_importance())
threshold = np.quantile(importances, 0.75)

mask = importances > threshold
print(f"Используем {mask.sum()} признаков из {len(mask)}")

X_train_red = X_train[:, mask]
X_val_red = X_val[:, mask]

# -------------------------------------------------------
# 4. Class weights (CatBoost справится лучше XGB)
# -------------------------------------------------------
cnt = Counter(y_train)
total = len(y_train)
class_weights = {cls: total / cnt[cls] for cls in cnt}

print("\nВеса классов:")
print(class_weights)

# -------------------------------------------------------
# 5. Финальная модель CatBoost
# -------------------------------------------------------
print("\n>>> Обучение финальной CatBoost модели...")

model = CatBoostClassifier(
    loss_function="MultiClass",
    eval_metric="TotalF1",
    iterations=1500,
    learning_rate=0.03,
    depth=7,
    l2_leaf_reg=3.0,
    random_seed=42,
    verbose=False,
    class_weights=class_weights
)

model.fit(
    X_train_red, y_train,
    eval_set=(X_val_red, y_val),
    verbose=False
)

# -------------------------------------------------------
# 6. Метрики
# -------------------------------------------------------
y_pred = model.predict(X_val_red).astype(int)
f1_macro = f1_score(y_val, y_pred, average="macro")

print("\n=== FINISHED ===")
print(f"Validation F1-macro: {f1_macro:.4f}\n")

print("=== Classification Report ===")
print(classification_report(y_val, y_pred))

# -------------------------------------------------------
# 7. Анализ ошибок
# -------------------------------------------------------
y_pred_vec = np.array(y_pred).reshape(-1)
y_val_vec  = np.array(y_val).reshape(-1)

errors_mask = (y_pred_vec != y_val_vec)

if num_errors > 0:
    y_proba = model.predict_proba(X_val_red)

    errors_df = pd.DataFrame({
        "Pubchem_ID": np.array(ids_val)[errors_mask],
        "true": y_val_vec[errors_mask],
        "pred": y_pred_vec[errors_mask],
        "max_proba": y_proba[errors_mask].max(axis=1)
    })

    print(errors_df.head(10))


>>> Быстрая модель CatBoost для отбора признаков...
Используем 525 признаков из 2099

Веса классов:
{np.int64(3): 5.1433021806853585, np.int64(0): 3.893867924528302, np.int64(5): 48.55882352941177, np.int64(1): 6.966244725738397, np.int64(7): 10.191358024691358, np.int64(4): 7.209606986899563, np.int64(6): 24.279411764705884, np.int64(2): 9.380681818181818}

>>> Обучение финальной CatBoost модели...

=== FINISHED ===
Validation F1-macro: 0.4717

=== Classification Report ===
              precision    recall  f1-score   support

           0       0.59      0.57      0.58       106
           1       0.38      0.36      0.37        59
           2       0.47      0.50      0.48        44
           3       0.43      0.40      0.41        81
           4       0.62      0.65      0.63        57
           5       0.50      0.44      0.47         9
           6       0.35      0.65      0.46        17
           7       0.40      0.35      0.37        40

    accuracy                   

In [9]:
# -------------------------------------------------------
# 8. Применение модели к тестовым данным и сохранение результата
# -------------------------------------------------------
print("\n>>> Применение модели к тестовым данным...")

# Убедимся, что используем те же признаки
X_test = test_df[feature_cols].values

# Применяем маску отбора признаков
X_test_red = X_test[:, mask]

# Предсказание
test_predictions = model.predict(X_test_red).astype(int).flatten()  # гарантируем 1D

# Извлекаем Pubchem_ID правильно
if "Pubchem_ID" not in test_df.columns:
    raise KeyError("Столбец 'Pubchem_ID' отсутствует в test_df!")

pubchem_ids = test_df["Pubchem_ID"].values
if pubchem_ids.ndim != 1:
    pubchem_ids = pubchem_ids.flatten()

# Проверка длины
assert len(pubchem_ids) == len(test_predictions), \
    f"Несовпадение длины: Pubchem_ID ({len(pubchem_ids)}) vs predictions ({len(test_predictions)})"

# Создание DataFrame
results_df = pd.DataFrame({
    "Pubchem_ID": pubchem_ids,
    "taste_cluster": test_predictions
})

# Сохранение
results_df.to_csv("sol.csv", index=False)

print(f"✅ Результаты сохранены в 'submission.csv'")
print(f"📌 Всего предсказаний: {len(results_df)}")
print("\nПервые 10 строк:")
print(results_df.head(10))


>>> Применение модели к тестовым данным...
✅ Результаты сохранены в 'submission.csv'
📌 Всего предсказаний: 888

Первые 10 строк:
   Pubchem_ID  taste_cluster
0       32594              2
1        7130              0
2        7455              7
3       31226              7
4       17000              0
5       61093              3
6     5354834              2
7       61457              1
8       80455              1
9      638122              4
